# Satellite Analysis

## Download images

Docs:  
https://developers.google.com/maps/documentation/maps-static/start

In [ ]:
import json
import numpy as np
import pandas as pd
import requests

from os import listdir, makedirs, path
from PIL import Image as PImage
from time import sleep

from env import GOOGLE_API_KEY

class obj(dict):
  __getattr__ = dict.get
  __setattr__ = dict.__setitem__
  __delattr__ = dict.__delitem__

def pad(n, l=3):
  return (l*"0" + str(n))[-l:] if n < 10**l else str(n)

def rnd(n, l=4):
  return float(round(n, l))

def to_obj(range_like):
  if "start" in range_like and "stop" in range_like:
    range_like = obj(range_like)
  if not (hasattr(range_like, "start") and hasattr(range_like, "start")):
    raise Exception("invalid parameter")
  return range_like

def range_step(range_like, l=4):
  range_like = to_obj(range_like)
  return rnd((range_like.stop - range_like.start) / range_like.count, l=l)

def range_space(range_like):
  range_like = to_obj(range_like)
  return np.linspace(range_like.start, range_like.stop, range_like.count, endpoint=False)

In [ ]:
MAPS_URL = "https://maps.googleapis.com/maps/api/staticmap"
IMG_SIZE = 512

MAPS_PARAMS = {
  "size": f"{IMG_SIZE}x{IMG_SIZE}",
  "scale": 1,
  "format": "jpg",
  "maptype": "satellite",
  "key": GOOGLE_API_KEY,
}

MAPS_HEADER = {
  "User-Agent": "Mozilla/5.0"
}

FOR = {
  "lat": { "start": -3.910, "stop": -3.710, "count": 32 }, # 32 OR 48 OR 64
  "lon": { "start": -38.655, "stop": -38.455, "count": 32 },
}

lat_cnt = FOR["lat"]["count"]
lon_cnt = FOR["lon"]["count"]
run_name = f"sat_{lon_cnt}x{lat_cnt}"

FOR["lat"]["step"] = range_step(FOR["lat"], 8)
FOR["lon"]["step"] = range_step(FOR["lon"], 8)
FOR["name"] = run_name

IMGS_DIR = "./imgs"
DATA_DIR = "./data"
SAT_IMGS_DIR = f"{IMGS_DIR}/{run_name}"

makedirs(SAT_IMGS_DIR, exist_ok=True)

with open(f"{SAT_IMGS_DIR}/{run_name}.json", "w") as ofp:
  json.dump(FOR, ofp, indent=2, ensure_ascii=False)

In [ ]:
img_data = []
for y,lat_start in enumerate(range_space(FOR["lat"])):
  for x,lon_start in enumerate(range_space(FOR["lon"])):
    img_id = f"{pad(x, 2)}_{pad(y, 2)}"
    row = {
      "lat_center": rnd(lat_start + FOR["lat"]["step"] * 0.5, 5),
      "lon_center": rnd(lon_start + FOR["lon"]["step"] * 0.5, 5),
      "x": x,
      "y": y,
      "img_id": img_id
    }
    img_data.append(row)
    filepath = f"{SAT_IMGS_DIR}/{run_name}_{img_id}.jpg"

    if path.isfile(filepath):
      continue

    lat_stop = rnd(lat_start + FOR["lat"]["step"], 5)
    lon_stop = rnd(lon_start + FOR["lon"]["step"], 5)
    MAPS_PARAMS["visible"] = f"{lat_start},{lon_start}|{lat_stop},{lon_stop}"

    response = requests.get(MAPS_URL, headers=MAPS_HEADER, params=MAPS_PARAMS, stream=True)
    PImage.open(response.raw).save(filepath)
    sleep(0.25)

  pd.DataFrame(img_data).to_csv(f"{DATA_DIR}/{run_name}.csv", index=False)

### Combine Images

In [ ]:
nimgs = 32
simg = 100

img_prefix = f"sat_{nimgs}x{nimgs}"
img_dir = f"imgs/{img_prefix}"

fimg = PImage.new("RGB", (nimgs*simg, nimgs*simg))

for i in range(nimgs):
  ii = pad(i, 2)
  for j in range(nimgs):
    jj = pad(nimgs - j - 1, 2)
    img = PImage.open(f"{img_dir}/{img_prefix}_{ii}_{jj}.jpg").resize((simg, simg))
    fimg.paste(img, (i*simg, j*simg))

fimg.save(f"imgs/{img_prefix}.jpg")
fimg

## CV Analysis